In [1]:
print("Hello Wold!")

Hello Wold!


In [2]:
print("Bye World!")

Bye World!


In [3]:
from pyspark import SparkContext, SparkConf

conf = SparkConf().setAppName("MyApp").setMaster("local[*]")
sc = SparkContext(conf=conf)

lines = sc.textFile("ml-100k/u.data")
ratings = lines.map(lambda line: line.split()[2])
results = ratings.countByValue()

for rating, count in sorted(results.items(), key=lambda x: int(x[0]), reverse=True):
    print(f"Rating: {rating}, Count: {count}")

sc.stop()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 15:30:36 WARN Utils: Your hostname, LAPTOP-OVHADNBM, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/29 15:30:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 15:30:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Rating: 5, Count: 21201
Rating: 4, Count: 34174
Rating: 3, Count: 27145
Rating: 2, Count: 11370
Rating: 1, Count: 6110


In [4]:
from prettytable import PrettyTable
from pyspark import SparkContext, SparkConf


conf = SparkConf().setAppName("FakeFriends").setMaster("local[*]")
sc = SparkContext(conf=conf)

def parseLine(line):
    fields = line.split(",")
    age = int(fields[2])
    numFriends = int(fields[3])
    return (age, numFriends)

lines = sc.textFile("fakefriends.csv")
rdd = lines.map(parseLine)


data = rdd.collect()

table = PrettyTable()
table.field_names = ["Age", "Number of Friends"]

for age, numFriends in data:
    table.add_row([age, numFriends])
print("Fake Friends Data:")

print(table)

totalsByAge = rdd.mapValues(lambda x: (x, 1)).reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))

averageByAge = totalsByAge.mapValues(lambda x: x[0] / x[1])

totalsData = totalsByAge.collect()
totalsTable = PrettyTable()
totalsTable.field_names = ["Age", "Total Number of Friends", "Count", "Average Number of Friends"]

for age, (totalFriends, count) in totalsData:
    avgeByAge = totalFriends / count
    totalsTable.add_row([age, totalFriends, count, avgeByAge])
print("Total Number of Friends by Age:")
print(totalsTable)


sc.stop()

Fake Friends Data:
+-----+-------------------+
| Age | Number of Friends |
+-----+-------------------+
|  33 |        385        |
|  33 |         2         |
|  55 |        221        |
|  40 |        465        |
|  68 |         21        |
|  17 |         1         |
+-----+-------------------+


Total Number of Friends by Age:
+-----+-------------------------+-------+---------------------------+
| Age | Total Number of Friends | Count | Average Number of Friends |
+-----+-------------------------+-------+---------------------------+
|  40 |           465           |   1   |           465.0           |
|  68 |            21           |   1   |            21.0           |
|  33 |           387           |   2   |           193.5           |
|  55 |           221           |   1   |           221.0           |
|  17 |            1            |   1   |            1.0            |
+-----+-------------------------+-------+---------------------------+


In [5]:
from pyspark import SparkContext, SparkConf
from prettytable import PrettyTable

conf = SparkConf().setAppName("MaxTemp").setMaster("local[*]")
sc = SparkContext(conf=conf)

def parseLine(line):
    fields = line.split(",")
    stationID = fields[0]
    entryType = fields[2]
    temperature = float(fields[3])
    return (stationID, entryType, temperature)

lines = sc.textFile("weatherData1800s.csv")
parsedLines = lines.map(parseLine)
maxTemps = parsedLines.filter(lambda x: 'TMAX' in x[1])
stationTemps = maxTemps.map(lambda x: (x[0], x[2]))
maxTempsByStation = stationTemps.reduceByKey(lambda x, y: max(x, y))
results = maxTempsByStation.collect()

table = PrettyTable()
table.field_names = ["Station ID", "Max Temperature"]

for stationID, maxTemp in results:
    table.add_row([stationID, maxTemp])

print("Maximum Temperatures by Station:")
print(table)

sc.stop() #stops the context


Maximum Temperatures by Station:
+-------------+-----------------+
|  Station ID | Max Temperature |
+-------------+-----------------+
| ITE00100554 |      323.0      |
| EZE00100082 |      323.0      |
+-------------+-----------------+
